[![session-10](Sesion10_banner.png)](https://github.com/semilleroCV/Hands-on-Computer-Vision/tree/main/sesiones/sesion10)

# <font color="EB9A54"><center> **Hands-on Sesión 10: Thermal Imaging Forward Model 📚🌡️** </center></font>

En este notebook construiremos paso a paso el modelo directo de una cámara térmica. Partiremos de la radiación emitida por los objetos de la escena, incorporaremos la propagación atmosférica y terminaremos con una lectura de sensor afectada por ruido gaussiano.

## <font color='#4C5FDA'> **Contenido**</font>

[**1. Cargar los datos de la escena**](#tema1)

[**2. Emisividad y radiancia espectral emitida por el objeto**](#tema2)

[**3. Atenuación y transmitancia atmosférica**](#tema3)

[**4. Radiancia espectral transmitida por la atmósfera**](#tema4)

[**5. Emisión del aire y modelo directo completo**](#tema5)

[**6. Ruido gaussiano del sensor**](#tema6)

# <font color="4C5FDA"> **1. Cargar los datos de la escena** </font><a id="tema1"></a>

La escena contiene mapas de temperatura, profundidad y emisividad espectral. También utilizaremos perfiles atmosféricos producidos por un simulador para varios gases y para la mezcla completa o envolvente.

In [ ]:
#@title **Cargar librerías y descargar los datos**
%matplotlib inline

from pathlib import Path

import gdown
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
from scipy.constants import c, h, k

folder_id = "1d150jOzFGBY0eea_Ak1kgLSU70OnwaC7"
data_dir = Path("data")
envelope_filename = "enclosure"
required_files = [
    "depth.npy",
    "emap.npy",
    "emissivity.npy",
    "matName_FullDatabase.npy",
    "tmap.npy",
    "wavelengths.npy",
    "atmospheres/CH4.txt",
    "atmospheres/CO2.txt",
    "atmospheres/H2O.txt",
    "atmospheres/O3.txt",
    f"atmospheres/{envelope_filename}.txt",
]

if not all((data_dir / filename).exists() for filename in required_files):
    gdown.download_folder(id=folder_id, output="./", quiet=False)

missing_files = [filename for filename in required_files if not (data_dir / filename).exists()]
if missing_files:
    raise FileNotFoundError(f"La descarga está incompleta. Faltan: {missing_files}")

print(f"Datos disponibles en: {data_dir.resolve()}")

In [ ]:
#@title **Cargar la escena**
wavelengths = np.load(data_dir / "wavelengths.npy")
depth_map = np.load(data_dir / "depth.npy")
temperature_map = np.load(data_dir / "tmap.npy")
material_map = np.load(data_dir / "emap.npy")
emissivity_cube = np.load(data_dir / "emissivity.npy").transpose(2, 0, 1)

material_names = np.load(
    data_dir / "matName_FullDatabase.npy", allow_pickle=True
).item()["matName"]
material_names = np.hstack(material_names.squeeze())

assert emissivity_cube.shape == (len(wavelengths), *temperature_map.shape)
assert depth_map.shape == temperature_map.shape == material_map.shape

print(f"Bandas espectrales: {len(wavelengths)} ({wavelengths[0]:.3f}–{wavelengths[-1]:.3f} µm)")
print(f"Resolución espacial: {temperature_map.shape[1]} × {temperature_map.shape[0]} píxeles")
print(f"Datos de emisividad espectral: {emissivity_cube.shape}")

# <font color="4C5FDA"> **2. Emisividad y radiancia espectral emitida por el objeto** </font><a id="tema2"></a>

La **emisividad** $\varepsilon(\lambda)$ describe qué fracción de la radiación de un cuerpo negro emite cada material. Es una propiedad espectral: dos materiales a la misma temperatura pueden producir firmas diferentes.

Como la ley de Planck ya se estudió en el notebook anterior, aquí reutilizaremos directamente la función $B(\lambda;T)$. La radiancia emitida por el objeto es

$$
L_{\mathrm{object}}(\lambda)
=\varepsilon(\lambda)\,B(\lambda;T_{\mathrm{object}}).
$$

In [ ]:
def blackbody(wavelength_um, temperature_k):
    # Radiancia de cuerpo negro en µW/(cm²·sr·µm).
    wavelength_m = np.asarray(wavelength_um, dtype=np.float64) * 1e-6
    temperature_k = np.asarray(temperature_k, dtype=np.float64)
    exponent = h * c / (wavelength_m * k * temperature_k)
    radiance = (2 * h * c**2) / (wavelength_m**5 * np.expm1(exponent))
    return radiance * 1e-4


def show_cube_bands(cube, title, cmap="inferno", band_indices=(0, 24, 48)):
    # Mostrar tres cortes de un arreglo espectral (bandas, alto, ancho).
    fig, axes = plt.subplots(1, len(band_indices), figsize=(18, 5), constrained_layout=True)
    for axis, band in zip(axes, band_indices):
        image = axis.imshow(cube[band], cmap=cmap)
        axis.set_title(f"{wavelengths[band]:.3f} µm")
        axis.axis("off")
        fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    fig.suptitle(title, fontsize=16)
    plt.show()

In [ ]:
#@title **Visualizar los materiales y la emisividad**
pixel_x, pixel_y = 960, 540
emissivity_band = 24
material_index = int(material_map[pixel_y, pixel_x])
material_name = material_names[material_index]

present_materials = np.unique(material_map)
material_palette = plt.colormaps["tab10"].resampled(len(present_materials))
material_cmap = ListedColormap([material_palette(i) for i in range(len(present_materials))])
material_norm = BoundaryNorm(np.arange(len(present_materials) + 1) - 0.5, material_cmap.N)
segmentation_map = np.searchsorted(present_materials, material_map)
legend_handles = [
    Patch(
        facecolor=material_cmap(index),
        label=f"{material_id}: {material_names[material_id]}",
    )
    for index, material_id in enumerate(present_materials)
]

fig, axes = plt.subplots(1, 3, figsize=(24, 6), constrained_layout=True)

axes[0].imshow(segmentation_map, cmap=material_cmap, norm=material_norm)
axes[0].scatter(pixel_x, pixel_y, color="red", s=45, marker="x")
axes[0].set_title("Mapa de materiales")
axes[0].set_xlabel("x (píxeles)")
axes[0].set_ylabel("y (píxeles)")
axes[0].grid(color="white", alpha=0.3, linewidth=0.6)
axes[0].legend(
    handles=legend_handles,
    title="Materiales",
    loc="upper left",
    bbox_to_anchor=(0, -0.12),
    ncol=2,
    fontsize=9,
)

image = axes[1].imshow(emissivity_cube[emissivity_band], cmap="viridis", vmin=0, vmax=1)
axes[1].scatter(pixel_x, pixel_y, color="red", s=45, marker="x")
axes[1].set_title(f"Mapa de emisividad a {wavelengths[emissivity_band]:.3f} µm")
axes[1].set_xlabel("x (píxeles)")
axes[1].set_ylabel("y (píxeles)")
axes[1].grid(color="white", alpha=0.3, linewidth=0.6)
fig.colorbar(image, ax=axes[1], label="Emisividad")

axes[2].plot(
    wavelengths,
    emissivity_cube[:, pixel_y, pixel_x],
    color="#4C5FDA",
    linewidth=2,
)
axes[2].set_title(f"Firma espectral del píxel ({pixel_x}, {pixel_y}) — {material_name}")
axes[2].set_xlabel("Longitud de onda (µm)")
axes[2].set_ylabel("Emisividad")
axes[2].set_ylim(0, 1)
axes[2].grid(alpha=0.25)

plt.show()

In [ ]:
#@title **Simular la radiancia espectral emitida por el objeto**
temperature_kelvin = temperature_map + 273.15
object_emission_cube = np.empty_like(emissivity_cube, dtype=np.float32)

for band, wavelength in enumerate(wavelengths):
    blackbody_band = blackbody(wavelength, temperature_kelvin)
    object_emission_cube[band] = emissivity_cube[band] * blackbody_band

print(f"Radiancia espectral emitida por el objeto: {object_emission_cube.shape}")
show_cube_bands(
    object_emission_cube,
    "Radiancia emitida por el objeto: ε(λ) B(λ; T_object)",
)

# <font color="4C5FDA"> **3. Atenuación y transmitancia atmosférica** </font><a id="tema3"></a>

La atmósfera reduce la radiancia del objeto de forma selectiva según la longitud de onda. Los archivos del simulador contienen la **transmitancia de referencia** $\tau$ y la **atenuación** $\alpha$ en dB para gases individuales y para la mezcla completa o envolvente.

Los perfiles atmosféricos tienen una resolución espectral diferente a la del sensor. Por ello, interpolaremos la transmitancia y la atenuación **directamente en cada valor de `wavelengths`**, garantizando una correspondencia uno a uno con las 49 bandas de la emisividad.

Para una distancia $d$, modelaremos la transmitancia como

$$
\tau(\lambda)=10^{-\alpha(\lambda)d/10}.
$$

In [ ]:
atmosphere_files = {
    "CH4": "CH4",
    "CO2": "CO2",
    "H2O": "H2O",
    "O3": "O3",
    "envolvente": envelope_filename,
}


def load_atmosphere(filename):
    # Cargar un perfil y remuestrearlo exactamente en las bandas del sensor.
    table = np.loadtxt(data_dir / "atmospheres" / f"{filename}.txt")

    if table.ndim != 2 or table.shape[1] < 3:
        raise ValueError(f"El perfil {filename} debe tener al menos tres columnas")

    native_wavelengths = table[:, 0]
    if not np.all(np.diff(native_wavelengths) > 0):
        raise ValueError(f"Las longitudes de onda de {filename} no están ordenadas")
    if wavelengths.min() < native_wavelengths.min() or wavelengths.max() > native_wavelengths.max():
        raise ValueError(f"Las bandas del sensor están fuera del rango de {filename}")

    spectral_mask = (
        (native_wavelengths >= wavelengths.min())
        & (native_wavelengths <= wavelengths.max())
    )
    native = table[spectral_mask]
    sensor_transmittance = np.interp(wavelengths, native_wavelengths, table[:, 1])
    sensor_attenuation_db = np.interp(wavelengths, native_wavelengths, table[:, 2])

    if sensor_transmittance.shape != wavelengths.shape:
        raise ValueError("La transmitancia no coincide con las bandas del sensor")
    if sensor_attenuation_db.shape != wavelengths.shape:
        raise ValueError("La atenuación no coincide con las bandas del sensor")

    return {
        "wavelength": native[:, 0],
        "transmittance": native[:, 1],
        "attenuation_db": native[:, 2],
        "sensor_transmittance": sensor_transmittance,
        "sensor_attenuation_db": sensor_attenuation_db,
    }


atmospheres = {
    label: load_atmosphere(filename)
    for label, filename in atmosphere_files.items()
}
print("Atmósferas cargadas:", ", ".join(atmospheres))
print(f"Cada perfil fue remuestreado en {len(wavelengths)} bandas del sensor")

In [ ]:
#@title **Comparar los gases y la mezcla atmosférica**
fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)

for name, profile in atmospheres.items():
    is_envelope = name == "envolvente"
    style = {"linewidth": 2.5, "color": "black"} if is_envelope else {"linewidth": 1.2, "alpha": 0.8}
    axes[0].plot(profile["wavelength"], profile["attenuation_db"], label=name, **style)
    axes[1].plot(profile["wavelength"], profile["transmittance"], label=name, **style)

axes[0].set_title("Atenuación espectral")
axes[0].set_xlabel("Longitud de onda (µm)")
axes[0].set_ylabel("Atenuación (dB por unidad de distancia)")
axes[0].set_yscale("symlog", linthresh=1e-3)
axes[0].grid(alpha=0.2)

axes[1].set_title("Transmitancia espectral de referencia")
axes[1].set_xlabel("Longitud de onda (µm)")
axes[1].set_ylabel("Transmitancia")
axes[1].set_ylim(-0.02, 1.02)
axes[1].grid(alpha=0.2)
axes[1].legend(ncol=2)

plt.show()

# <font color="4C5FDA"> **4. Radiancia espectral transmitida por la atmósfera** </font><a id="tema4"></a>

Usaremos la envolvente como la atmósfera completa. En este primer paso solo aplicaremos la transmitancia a la radiación del objeto:

$$
L_{\mathrm{transmitted}}(\lambda)
=\tau(\lambda)L_{\mathrm{object}}(\lambda).
$$

El mapa de profundidad aparece ahora porque la distancia recorrida determina cuánto se atenúa cada píxel.

In [ ]:
#@title **Calcular la transmitancia para toda la escena**
selected_atmosphere = "envolvente"
attenuation_db = atmospheres[selected_atmosphere]["sensor_attenuation_db"]

transmittance_cube = np.empty_like(emissivity_cube, dtype=np.float32)
for band, attenuation_band in enumerate(attenuation_db):
    transmittance_cube[band] = 10 ** (-attenuation_band * depth_map / 10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
depth_image = axes[0].imshow(depth_map, cmap="viridis")
axes[0].set_title("Profundidad de la escena (m)")
axes[0].axis("off")
fig.colorbar(depth_image, ax=axes[0], label="Distancia (m)")

tau_band = 0
tau_image = axes[1].imshow(transmittance_cube[tau_band], cmap="magma", vmin=0, vmax=1)
axes[1].set_title(f"Transmitancia de la envolvente a {wavelengths[tau_band]:.3f} µm")
axes[1].axis("off")
fig.colorbar(tau_image, ax=axes[1], label="Transmitancia")
plt.show()

In [ ]:
#@title **Aplicar la transmitancia a la radiancia espectral emitida**
transmitted_object_cube = transmittance_cube * object_emission_cube

show_cube_bands(
    transmitted_object_cube,
    "Radiancia del objeto después de atravesar la atmósfera: τ ε B(λ; T_object)",
)

In [ ]:
#@title **Comparar el efecto de la transmitancia en un píxel**
plt.figure(figsize=(10, 5))
plt.plot(
    wavelengths,
    object_emission_cube[:, pixel_y, pixel_x],
    label="Emitida por el objeto",
    linewidth=2,
)
plt.plot(
    wavelengths,
    transmitted_object_cube[:, pixel_y, pixel_x],
    label="Después de la atmósfera",
    linewidth=2,
)
plt.xlabel("Longitud de onda (µm)")
plt.ylabel("Radiancia [µW/(cm²·sr·µm)]")
plt.title(f"Impacto de la transmitancia en el píxel ({pixel_x}, {pixel_y})")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

# <font color="4C5FDA"> **5. Emisión del aire y modelo directo completo** </font><a id="tema5"></a>

La atmósfera no solo atenúa: también emite radiación. Supondremos una atmósfera homogénea, en equilibrio térmico local y con temperatura conocida $T_{\mathrm{air}}$. Por la ley de Kirchhoff, su emisividad es $1-\tau$; por tanto,

$$
L_{\mathrm{air}}(\lambda)
=\left[1-\tau(\lambda)\right]B(\lambda;T_{\mathrm{air}}).
$$

El modelo directo completo queda

$$
\boxed{
L_{\mathrm{aperture}}(\lambda)
=\underbrace{\tau(\lambda)\,\varepsilon(\lambda)\,B(\lambda;T_{\mathrm{object}})}_{\text{objeto transmitido}}
+\underbrace{[1-\tau(\lambda)]B(\lambda;T_{\mathrm{air}})}_{\text{emisión del aire}}
}.
$$

In [ ]:
#@title **Simular la emisión atmosférica y la radiancia espectral en la apertura**
air_temperature_k = 288.0
air_blackbody = blackbody(wavelengths, air_temperature_k).astype(np.float32)

atmospheric_emission_cube = (
    (1 - transmittance_cube) * air_blackbody[:, None, None]
).astype(np.float32)

aperture_cube = transmitted_object_cube + atmospheric_emission_cube

print(f"Temperatura del aire: {air_temperature_k:.1f} K")
print(f"Radiancia espectral en la apertura: {aperture_cube.shape}")

In [ ]:
#@title **Visualizar los términos del modelo directo**
display_band = 0
terms = [
    (object_emission_cube[display_band], "1. Objeto emitido: ε B(λ; T_object)"),
    (transmitted_object_cube[display_band], "2. Objeto transmitido: τ ε B(λ; T_object)"),
    (atmospheric_emission_cube[display_band], "3. Emisión del aire: (1-τ) B(λ; T_air)"),
    (aperture_cube[display_band], "Resultado en la apertura"),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)
for axis, (term, title) in zip(axes.flat, terms):
    image = axis.imshow(term, cmap="inferno")
    axis.set_title(title)
    axis.axis("off")
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
fig.suptitle(f"Modelo directo a {wavelengths[display_band]:.3f} µm", fontsize=16)
plt.show()

In [ ]:
#@title **Comparar las contribuciones espectrales en un píxel**
plt.figure(figsize=(10, 5))
plt.plot(wavelengths, transmitted_object_cube[:, pixel_y, pixel_x], label="Objeto transmitido")
plt.plot(wavelengths, atmospheric_emission_cube[:, pixel_y, pixel_x], label="Emisión del aire")
plt.plot(wavelengths, aperture_cube[:, pixel_y, pixel_x], label="Total en la apertura", linewidth=2.5)
plt.xlabel("Longitud de onda (µm)")
plt.ylabel("Radiancia [µW/(cm²·sr·µm)]")
plt.title(f"Composición espectral en el píxel ({pixel_x}, {pixel_y})")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

# <font color="4C5FDA"> **6. Ruido gaussiano del sensor** </font><a id="tema6"></a>

Para finalizar, aproximaremos la incertidumbre electrónica del sensor mediante ruido gaussiano aditivo, independiente entre píxeles y bandas:

$$
L_{\mathrm{measured}}(\lambda)=L_{\mathrm{aperture}}(\lambda)+n(\lambda),
\qquad n(\lambda)\sim\mathcal{N}(0,\sigma^2).
$$

Este modelo todavía no incluye la respuesta espectral, integración, ganancia ni cuantización de un detector específico.

In [ ]:
#@title **Simular la lectura ruidosa del sensor**
noise_fraction = 0.01
noise_std = noise_fraction * np.percentile(aperture_cube, 99)
rng = np.random.default_rng(314)

sensor_cube = np.empty_like(aperture_cube, dtype=np.float32)
for band in range(len(wavelengths)):
    noise = rng.normal(0, noise_std, size=aperture_cube[band].shape)
    sensor_cube[band] = aperture_cube[band] + noise

rmse = np.sqrt(np.mean((sensor_cube - aperture_cube) ** 2))
mae = np.mean(np.abs(sensor_cube - aperture_cube))

print(f"Desviación estándar del ruido: {noise_std:.4f} µW/(cm²·sr·µm)")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

In [ ]:
#@title **Comparar la radiancia ideal y la medición**
sensor_band = 24
noise_image = sensor_cube[sensor_band] - aperture_cube[sensor_band]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
images = [
    (aperture_cube[sensor_band], "Radiancia ideal", "inferno"),
    (sensor_cube[sensor_band], "Medición con ruido", "inferno"),
    (noise_image, "Ruido añadido", "coolwarm"),
]

for axis, (image_data, title, cmap) in zip(axes, images):
    image = axis.imshow(image_data, cmap=cmap)
    axis.set_title(f"{title} — {wavelengths[sensor_band]:.3f} µm")
    axis.axis("off")
    fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)

plt.show()

In [ ]:
#@title **Visualizar el impacto del ruido electrónico en un píxel**
plt.figure(figsize=(10, 5))
plt.plot(
    wavelengths,
    aperture_cube[:, pixel_y, pixel_x],
    label="Radiancia ideal",
    linewidth=2,
)
plt.plot(
    wavelengths,
    sensor_cube[:, pixel_y, pixel_x],
    label="Medición con ruido",
    linewidth=1.5,
)
plt.xlabel("Longitud de onda (µm)")
plt.ylabel("Radiancia [µW/(cm²·sr·µm)]")
plt.title(f"Impacto del ruido electrónico del sensor en el píxel ({pixel_x}, {pixel_y})")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## Resultado

La radiancia espectral final incorpora, en orden, la emisividad y temperatura del objeto, la transmitancia dependiente de la atmósfera y la distancia, la emisión térmica del aire y el ruido gaussiano del sensor.